# From Rydberg atoms to a shift plan
# QoolQit and QUBO Solver in practice

### Pasqal Workshop
#### Trainers: Vincent Martin, Krisztian Benyo

In this tutorial you will:

1. Get to know **QUBO** problems and why they map naturally onto **neutral-atom (Rydberg) quantum hardware**.
2. Meet **[QoolQit](https://github.com/pasqal-io/qoolqit)** (the low-level toolbox for the Rydberg analog model) and
   **[QUBO Solver](https://github.com/pasqal-io/qubo-solver)** (the high-level pipeline that solves QUBO type optimisation problems).
3. Solve a small QUBO *by hand* with QoolQit (embedding, drive, program, emulation).
4. Solve a QUBO with QUBO Solver, both with the one-line `Solver` shortcut and with the step-by-step functional API.
5. Build a **minimum working example of an industrial pipeline**: choosing which maintenance jobs to run during
   one factory shift when jobs conflict over technicians and safety zones.

> **Sources.** The QoolQit part follows the official tutorial
> [*Solving a basic QUBO problem*](https://docs.pasqal.com/qoolqit/tutorials/solving_a_qubo/).
>
> The QUBO Solver part follows the
> [quantum pipeline user guide](https://pasqal-io.github.io/qubo-solver/latest/user_guide/quantum/intro/) and the
> [*Solving a QUBO problem flexibly*](https://pasqal-io.github.io/qubo-solver/latest/tutorials/02-qubosolver-in-full/) tutorials.

## Setup

Everything runs **locally on an emulator** - no quantum hardware or cloud account is needed.
Use a fresh virtual environment (Python 3.10+; it is recommended to have a recent 3.11 or 3.12 Python version).

In [ ]:
import itertools
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

Please do keep in mind that both QoolQit and QUBO solver libraries are young and evolving constantly.

---
# 1. Introduction

## 1.1 What is a QUBO?

A **Quadratic Unconstrained Binary Optimization** (QUBO) instance on $N$ binary variables is defined by a symmetric matrix
$Q \in \mathbb{R}^{N\times N}$. *Solving* it means finding the bitstring $z \in \{0,1\}^N$ that minimizes

$$
f(z) = z^\top Q z = \sum_i Q_{ii}\, z_i + \sum_{i \neq j} Q_{ij}\, z_i z_j .
$$

* The **diagonal** $Q_{ii}$ is the cost (negative = reward) of switching variable $i$ on.
* The **off-diagonal** $Q_{ij}$ is the cost of switching $i$ and $j$ on *together* (positive = they compete with each other).
* Because $z_i^2 = z_i$ for bits, linear terms live on the diagonal, and constants can be dropped.
* QUBOs are **scale-invariant**: multiplying $Q$ by a positive number does not change the optimal bitstring.
* Many hard combinatorial problems (maximum independent set, max-cut, scheduling, portfolio selection, ...) can be written
  as QUBOs. The worst-case complexity is exponential, as there are $2^N$ candidate bitstrings.

## 1.2 Why neutral atoms? The Rydberg mapping

In a **neutral-atom quantum processor**, each qubit is an atom held in an optical tweezer. When an atom is excited to a
*Rydberg state* (occupation $n_i = 1$), it strongly **repels** its neighbours: the interaction energy $J_{ij}$ falls off
quickly with the distance between atoms $i$ and $j$ (in QoolQit's dimensionless units, roughly following $1/r_{ij}^6$).

With a laser **detuning** $\delta_i$ rewarding excitation, the classical (zero-amplitude) energy is

$$
H = -\sum_i \big(\tfrac{\Omega}{2}\,\sigma^x_i - \delta_i\, n_i\big) \;+\; \sum_{i<j} J_{ij}\, n_i n_j .
$$

Comparing with $f(z)$ gives the dictionary used by both libraries:

$$
Q_{ii} \;\longleftrightarrow\; -2\,\delta_i \quad\text{(drive shaping)},
\qquad
Q_{ij} \;\longleftrightarrow\; J_{ij} \quad\text{(embedding)} .
$$

(The factor 2 appears because the symmetric $Q$ counts every pair twice; in fact $H(z) = f(z)/2$.)
The recipe to solve a QUBO is therefore:

1. **Embed:** place atoms in the plane so that pairwise interactions match $Q_{ij}$.
2. **Shape the drive:** choose time-dependent laser amplitude $\Omega(t)$ and detuning $\delta(t)$ - an *annealing*
   schedule that slowly drives the system towards the ground state of $H$.
3. **Run and measure:** sample many bitstrings (shots); the most frequent low-cost ones are your candidate solutions.

Because Rydberg interactions are *repulsive*, QUBOs with **non-negative off-diagonal terms** are a particularly natural fit. Our industrial example in Section 4 is of exactly this type.

## 1.3 QoolQit - the toolbox for the Rydberg analog model

[QoolQit](https://github.com/pasqal-io/qoolqit) is a Python library for **algorithm development in the Rydberg analog
model**. It gives you the building blocks to describe and simulate analog quantum programs in dimensionless units:

| Building block | What it is | 
|---|---|
| `Register` | Positions of the atoms (e.g. built from an embedded graph with `Register.from_graph`) |
| Waveforms (`Ramp`, `PiecewiseLinear`, ...) | Time-dependent functions for amplitude and detuning |
| `Drive` | Pairs an amplitude waveform with a detuning waveform |
| `QuantumProgram` | A register plus a drive: the object you run |
| `AnalogDevice`, `AnalogDeviceWithDMM` | Device constraints a program must satisfy (`program.compile_to(device)`) |
| `qoolqit.embedding` (e.g. `InteractionEmbedder`) | Turns a matrix into a graph with node coordinates |
| `qoolqit.execution.LocalEmulator` | Runs a compiled program on a local emulator and returns bitstring counts |

QoolQit is *general purpose*: you control every step. That is particularly suitable for research and for learning,
but you must make all design choices yourself (Section 2).

## 1.4 QUBO Solver - the pipeline on top

[QUBO Solver](https://github.com/pasqal-io/qubo-solver) is a library for solving QUBOs on Pasqal's neutral-atom
processors. It **wraps the three steps above** (and uses QoolQit under the hood), so you can go from matrix to solution
in a few lines. By default it runs on a local emulator.

**Quantum pipeline** (each step is also available as a standalone function):

| Step | Module | Examples |
|---|---|---|
| 1. Embedding | `qubosolver.embedding` | `blade`, `greedy` |
| 2. Drive shaping | `qubosolver.drive_shaping` | `proportional_diagonal`, local energy scale, Bayesian search |
| 3. Compile and run | `qubosolver.solving`, backends | `LocalEmulator`, `RemoteEmulator`, `QPU` |

**Classical pipeline:** classical solvers (such as tabular search) are available as well, which is useful for establishing baselines.

**Two ways to use it**

* *Object API:* `Solver(instance, SolverConfig(...)).solve()` - one call, sensible defaults.
* *Functional API:* call `embedding`, `drive_shaping`, `solving` yourself - more insight and control.

Results come back as a `Solution` object that holds the candidate bitstrings with their costs, counts and probabilities,
and `analysis.to_dataframe([solution])` turns it into a pandas table.

| | QoolQit | QUBO Solver |
|---|---|---|
| Level | low-level, general Rydberg toolbox | high-level, QUBO-specific |
| You choose | positions, waveforms, device | (optionally) embedding, drive-shaping method, backend |
| Input | `Register` + `Drive` | a QUBO matrix (`Instance`) |
| Output | bitstring counts | `Solution` with costs and probabilities |

### Helper: classical brute force

For small instances we can check any quantum result by **exhaustive enumeration** of all $2^N$ bitstrings.

In [ ]:
def brute_force(Q, k=2):
    """Evaluate all 2^N bitstrings; return the k best as (list of bitstrings, array of costs)."""
    Q = np.asarray(Q, dtype=float)
    n = len(Q)
    all_bits = np.array(list(itertools.product([0, 1], repeat=n)))
    costs = np.einsum("ki,ij,kj->k", all_bits, Q, all_bits)
    order = np.argsort(costs, kind="stable")[:k]
    return ["".join(map(str, all_bits[i])) for i in order], costs[order]


def plot_distribution(counter, solutions, bins=10, title=None):
    """Bar chart of the `bins` most frequent bitstrings; known optimal solutions in green."""
    top = dict(Counter(counter).most_common(bins))
    colors = ["tab:green" if key in set(solutions) else "tab:blue" for key in top]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(range(len(top)), list(top.values()), color=colors, tick_label=list(top.keys()))
    ax.set_xlabel("Bitstring")
    ax.set_ylabel("Counts")
    ax.tick_params(axis="x", rotation=60)
    if title:
        ax.set_title(title)
    fig.tight_layout()
    return fig

### The QUBO cost function

The `qubo_cost(Q, z)` function returns $f(z) = z^\top Q z$ for a bitstring `z` (a sequence of 0/1).

In [ ]:
def qubo_cost(Q, z):
    """Return f(z) = z^T Q z."""
    Q = np.asarray(Q, dtype=float)
    z = np.asarray(z, dtype=float)
    cost = z @ Q @ z
    return float(cost)

### Exercise

Check that the cost function calculator functions as intended in the following example

In [ ]:
Q_toy = np.array([[-1.0, 2.0],
                  [2.0, -1.0]])

### From QUBO to Rydberg parameters

The `qubo_to_rydberg(Q)` function calculates the local detuning values
$\tilde\delta_i = -Q_{ii}/2$ and the pairwise interaction terms $\tilde J_{ij} = Q_{ij}$ for $i<j$.

In [ ]:
def qubo_to_rydberg(Q):
    """Return (delta, J): local detunings (vector) and pair interactions (strictly upper-triangular matrix)."""
    Q = np.asarray(Q, dtype=float)
    delta = -np.diag(Q) / 2
    J = np.triu(Q, k=1)
    return delta, J

---
# 2. Example 1 - Solving a QUBO with QoolQit

We follow the official QoolQit tutorial on a 5-variable instance. Here we make every design choice ourselves.

## 2.1 The problem and its classical solution

The matrix is already given. We normalise it
(QUBOs are scale-invariant) and compute the optimum classically for later comparison.

In [ ]:
Q = np.array([
    [-10.0,       19.7365809,  19.7365809,  5.42015853, 5.42015853],
    [19.7365809, -10.0,        20.67626392, 0.17675796, 0.85604541],
    [19.7365809,  20.67626392, -10.0,       0.85604541, 0.17675796],
    [5.42015853,  0.17675796,  0.85604541, -10.0,       0.32306662],
    [5.42015853,  0.85604541,  0.17675796,  0.32306662, -10.0],
])

# QUBOs are scale-invariant: work with the normalised matrix.
Q = Q / Q.max()

best_bits, best_costs = brute_force(Q, k=2)
marked_bitstrings = best_bits
print("Two best solutions:", best_bits)
print("Respective costs:  ", np.round(best_costs, 4))

## 2.2 Embedding: from matrix to atom positions

The `InteractionEmbedder` maps the matrix to a **graph with node coordinates**, so that the pairwise distances reproduce
the off-diagonal entries $Q_{ij}$. From that graph we create the qubit `Register`.

In [ ]:
from qoolqit import Drive, PiecewiseLinearWaveform, QuantumProgram, RampWaveform, Register, AnalogDevice
from qoolqit.embedding import InteractionEmbedder
from qoolqit.execution import LocalEmulator

embedder = InteractionEmbedder()
embedded_graph = embedder.embed(Q)
embedded_graph.draw()

register = Register.from_graph(embedded_graph)

## 2.3 The drive: an annealing schedule

We raise the amplitude $\Omega$ to a plateau value, hold it, and lower it again, while **sweeping the detuning** linearly
from $\delta_i = -2\Omega$ to $\delta_f = +2\Omega$ over a total duration $T$. We take $\Omega$ as the median of the
positive entries of $Q$. A long enough duration lets the system follow the schedule adiabatically.

In [ ]:
omega = np.median(Q[Q > 0].flatten())
delta_i = -2.0 * omega
delta_f = 2.0 * omega
T = 52.0

wf_amp = PiecewiseLinearWaveform([T / 4, T / 2, T / 4], [0.0, omega, omega, 0.0])
wf_det = RampWaveform(T, delta_i, delta_f)

drive = Drive(amplitude=wf_amp, detuning=wf_det)

## 2.4 Program, compilation and emulation

A `QuantumProgram` pairs the register with the drive. Before running it we **compile it to a device**, which checks the
program against the device's constraints (and rescales it to physical units). We then sample 500 shots on the local
emulator.

In [ ]:
program = QuantumProgram(register, drive)
program.draw()

program.compile_to(device=AnalogDevice())

emulator = LocalEmulator(num_shots=500)

# execute the program on the local emulator
job = emulator.run(program)

# get the results
results = job.results()

counter = results.final_bitstrings
print(counter)

In [ ]:
fig = plot_distribution(counter, marked_bitstrings, title="QoolQit: sampled bitstrings (green = classical optimum)")
plt.show()

### Exercise - success probability

Compute the **fraction of shots** that returned one of the classically optimal bitstrings in `marked_bitstrings`.
(Help: `counter` behaves like a dictionary from bitstring to count; missing keys count as 0.)

In [ ]:
n_shots = sum(counter.values())
p_opt = ...  # <-- YOUR CODE HERE: share of shots that hit a bitstring in marked_bitstrings

assert p_opt is not Ellipsis, "Fill in the blank line first"
print(f"Probability of sampling an optimal solution: {p_opt:.1%}")
assert 0.0 <= p_opt <= 1.0

The two optimal bitstrings dominate the histogram, i.e. the annealing
schedule worked. Emulator results are sampled, so your exact counts will differ slightly.

---
# 3. Industrial mini-pipeline: choosing maintenance jobs for one shift

## 3.1 The business problem

A plant has **8 candidate maintenance jobs** for the next shift. Each job has a *value* (risk reduction score if it is
done). Two jobs **cannot both be scheduled** if they

* take place in the **same zone** (safety lock-out: one crew per zone), or
* need the **same technician**.

**Goal:** choose the set of jobs with maximum total value that contains no conflicting pair. This is a (weighted)
**maximum independent set** problem on the "conflict graph" - a classic source of QUBOs, and the same structure appears in
sensor placement, task selection for scarce resources, frequency assignment, and many other settings.

## 3.2 QUBO formulation

With $z_i = 1$ if job $i$ is scheduled, $w_i$ its value, $E$ the set of conflicting pairs and $P$ a penalty:

$$
\min_z \; -\sum_i w_i z_i \;+\; P \sum_{(i,j)\in E} z_i z_j .
$$

* $P > \max_i w_i$ guarantees that dropping one of two conflicting jobs is always at least as good as keeping both,
  so optimal solutions are conflict-free. We use $P = 1.5\,\max_i w_i$.
* In matrix form: $Q_{ii} = -w_i$ and, **because $z^\top Q z$ counts each pair twice**, $Q_{ij} = Q_{ji} = P/2$ for
  $(i,j) \in E$.
* All off-diagonal entries are non-negative: a perfect match for repulsive Rydberg interactions.

## 3.3 Pipeline overview

`data -> conflict graph -> QUBO matrix -> exact reference -> QUBO Solver -> decode and validate -> report`

### Step 1: data and conflict graph

In [ ]:
jobs = pd.DataFrame(
    [
        # name,                       value, zone, technician
        ("Conveyor bearing swap",       8,   "A",  "T1"),
        ("Hydraulic press seal",        6,   "A",  "T2"),
        ("Boiler inspection",           9,   "B",  "T1"),
        ("Cooling pump overhaul",       7,   "B",  "T3"),
        ("Robot arm calibration",       5,   "C",  "T2"),
        ("Packaging sensor check",      4,   "C",  "T3"),
        ("Compressor filter change",    6,   "D",  "T1"),
        ("Paint booth ventilation",     5,   "D",  "T4"),
    ],
    columns=["job", "value", "zone", "technician"],
)
values = jobs["value"].to_numpy(dtype=float)
n_jobs = len(jobs)

# Two jobs conflict if they share a zone or a technician.
edges = [
    (i, j)
    for i, j in itertools.combinations(range(n_jobs), 2)
    if jobs.zone[i] == jobs.zone[j] or jobs.technician[i] == jobs.technician[j]
]

print(jobs.to_string(index=False))
print(f"{n_jobs} jobs, {len(edges)} conflicts: {edges}")

G = nx.Graph()
G.add_nodes_from(range(n_jobs))
G.add_edges_from(edges)
pos = nx.spring_layout(G, seed=3)
fig, ax = plt.subplots(figsize=(5, 4))
nx.draw_networkx(G, pos, ax=ax, node_color="lightsteelblue", node_size=650,
                 labels={i: f"J{i}\n({int(values[i])})" for i in G}, font_size=8)
ax.set_title("Conflict graph (node label: job id and value)")
ax.axis("off")
plt.show()

### Step 2: build the QUBO matrix

### Exercise - the penalty entries

The diagonal ($-w_i$) is given. Complete the off-diagonal entry for each conflicting pair. Remember that
$z^\top Q z$ counts every pair **twice** (once as $Q_{ij}$, once as $Q_{ji}$), so each of the two symmetric entries must
carry **half** of the penalty $P$.

In [ ]:
def build_qubo(values, edges, penalty):
    """QUBO for: min  -sum_i w_i z_i + penalty * sum_{(i,j) in edges} z_i z_j   (as z^T Q z)."""
    n = len(values)
    Q = np.zeros((n, n))
    for i in range(n):
        Q[i, i] = -values[i]
    for i, j in edges:
        Q[i, j] = Q[j, i] = ...  # <-- YOUR CODE HERE: half of the penalty on each symmetric entry
    return Q


penalty = 1.5 * values.max()
Q_job = build_qubo(values, edges, penalty)

print(penalty)

### Step 3: decode, validate, and an exact reference

Before using any solver we need two helpers: one to **count violated constraints** of a bitstring and one to
**evaluate** a bitstring in business terms (total value, number of conflicts). Then we compute the exact optimum by
brute force ($2^8 = 256$ candidates) as the ground truth.

### Exercise - count conflicts

Complete `count_conflicts(z, edges)`: the number of conflicting pairs $(i,j) \in E$ for which **both** jobs are selected.

In [ ]:
def count_conflicts(z, edges):
    """Number of pairs (i, j) in `edges` with z[i] = z[j] = 1."""
    n_conflicts = ...  # <-- YOUR CODE HERE
    return int(n_conflicts)


def evaluate(bits, values, edges):
    """Business-level evaluation of a bitstring such as '01001010'."""
    z = np.array(list(bits), dtype=int)
    return float(values @ z), count_conflicts(z, edges)

In [ ]:
assert count_conflicts([1, 1, 0, 0, 0, 0, 0, 0], edges) == 1     # jobs 0 and 1 share zone A
assert count_conflicts([1, 0, 0, 0, 0, 0, 0, 1], edges) == 0     # jobs 0 and 7 do not conflict
assert count_conflicts([1, 0, 1, 0, 0, 0, 1, 0], edges) == 3     # jobs 0, 2, 6 all need technician T1
print("Exercise passed")

In [ ]:
exact_bits, exact_costs = brute_force(Q_job, k=3)
print("Exact top-3 solutions (lowest QUBO cost first):")
for b, c in zip(exact_bits, exact_costs):
    value, n_conf = evaluate(b, values, edges)
    chosen = [jobs.job[i] for i, bit in enumerate(b) if bit == "1"]
    print(f"  {b}  cost={c:6.2f}  value={value:4.0f}  conflicts={n_conf}  -> {chosen}")

optimal_value = evaluate(exact_bits[0], values, edges)[0]
print("\nOptimal total value:", optimal_value)

### Step 4: solve with QoolQit

We now solve the job-selection QUBO **directly with QoolQit**, applying the recipe of Section 2 to the matrix `Q_job`.
Because QUBOs are scale-invariant we first **normalise** the matrix so its largest entry (in magnitude) is 1.

1. **Embedding.** `InteractionEmbedder` places one atom per job so that the interactions reproduce the off-diagonal penalties
   $Q_{ij}$ (conflicting jobs end up close together, compatible jobs far apart).
2. **Drive.** Our diagonal entries *differ* (they are the job values $-w_i$), so a purely global detuning is not enough.
   Following the dictionary $Q_{ii} \leftrightarrow -2\tilde\delta_i$ of Section 1.2, we add a **Detuning Map Modulator**
   whose per-atom weights are the normalised rewards $1 - w_i/\max w$ (the standard recipe for weighted problems in QoolQit):
   valuable jobs are excited more readily. The global schedule is `0 -> Omega -> 0` for the amplitude and
   `delta_0 -> 0 -> delta_f` for the detuning.
3. **Run.** Compile for an ideal `MockDevice`, emulate 1000 shots and convert the counts into a table with the **same columns**
   (`bitstrings`, `counts`, `probs`) as before, so that Steps 5 and 6 work unchanged.

#### 4a. Embedding and register

In [ ]:
from qoolqit import Register
from qoolqit.graphs import DataGraph
from qoolqit.embedding import SpringLayoutEmbedder

scale = np.abs(Q_job).max()
Q_norm = Q_job / scale                       # scale-invariant: same optimum

G_job = DataGraph.from_matrix(Q_norm)
job_graph = SpringLayoutEmbedder().embed(G_job)     # DataGraph with optimised coordinates
job_graph.draw()
job_register = Register.from_graph(job_graph)

#### 4b. Energy scales, detuning map and drive

In [ ]:
from qoolqit import ConstantWaveform, Drive, InterpolatedWaveform
from qoolqit.drive import DetuningMapModulator
from qoolqit import MockDevice, AnalogDeviceWithDMM, QuantumProgram

def min_distance(register, embedded_graph):
    """Smallest pairwise atom distance (falls back to the embedded coordinates)."""
    try:
        return float(register.min_distance())
    except Exception:
        pts = np.array(list(embedded_graph.coords.values()), dtype=float)
        d = np.linalg.norm(pts[:, None, :] - pts[None, :, :], axis=-1)
        return float(d[np.triu_indices(len(pts), k=1)].min())


J_max = 1.0 / min_distance(job_register, job_graph) ** 6   # strongest interaction (closest pair)
Omega = 0.5 * J_max            # drive amplitude, tailored to the interaction scale
delta_0 = -Omega                 # negative: all atoms in the ground state at t = 0
delta_f = -delta_0               # positive: excitations rewarded at t = T
T = 200.0                        # long schedule (ideal device)

# Detuning map from the diagonal: reward w_i = -Q_ii, normalised exactly as in the MWIS recipe.
rewards = -np.diag(Q_norm)
dmm_weights = 1.0 - rewards / rewards.max()
det_map = {q: float(dmm_weights[i]) for i, q in enumerate(job_register.qubits.keys())}
dmm = DetuningMapModulator(ConstantWaveform(T, -delta_f), det_map)

job_drive = Drive(
    amplitude=InterpolatedWaveform(T, [0.0, Omega, 0.0]),
    detuning=InterpolatedWaveform(T, [delta_0, 0.0, delta_f]),
    dmm=dmm,
)
job_program = QuantumProgram(job_register, job_drive)
job_program.draw()
job_program.compile_to(device=MockDevice())

#### 4c. Emulate and decode

Bit $i$ of a measured bitstring corresponds to job $i$ (atoms are created in node order, as in Section 2).

In [ ]:
try:
    job_emulator = LocalEmulator(num_shots=1000)
except TypeError:
    job_emulator = LocalEmulator()
job_res = job_emulator.run(job_program).results()
job_counts = Counter(job_res.final_bitstrings if hasattr(job_res, "final_bitstrings")
                     else job_res[0].final_bitstrings)
n_shots = sum(job_counts.values())

df_raw = pd.DataFrame(
    [{"bitstrings": bits, "counts": c, "probs": c / n_shots} for bits, c in job_counts.most_common()]
)
print(df_raw.head(8))

### Step 5: post-processing - validate and repair

A quantum sampler returns a *distribution*. In an industrial setting we never trust a raw sample blindly:

1. re-evaluate each candidate in business terms,
2. keep only **feasible** candidates (no conflicts) - or **repair** infeasible ones greedily by dropping the least valuable
   job of each violated pair,
3. report the best plan together with its quality relative to the known optimum.

In [ ]:
def repair(bits, values, edges):
    """Greedy repair: while a conflict exists, drop the least valuable job involved in it."""
    z = np.array(list(bits), dtype=int)
    while True:
        violated = [(i, j) for i, j in edges if z[i] and z[j]]
        if not violated:
            return "".join(map(str, z))
        i, j = violated[0]
        z[i if values[i] <= values[j] else j] = 0


cand = df_raw.copy()
cand["bits"] = cand["bitstrings"].astype(str).str.zfill(n_jobs)
evals = [evaluate(b, values, edges) for b in cand["bits"]]
cand["value"] = [e[0] for e in evals]
cand["conflicts"] = [e[1] for e in evals]
cand["feasible"] = cand["conflicts"] == 0
cand["repaired_bits"] = [repair(b, values, edges) for b in cand["bits"]]
cand["repaired_value"] = [evaluate(b, values, edges)[0] for b in cand["repaired_bits"]]

print(cand[["bits", "counts", "probs", "value", "conflicts", "feasible", "repaired_value"]]
      .sort_values("probs", ascending=False).head(10).to_string(index=False))

feasible_mass = cand.loc[cand.feasible, "probs"].sum()
best = cand.sort_values("repaired_value", ascending=False).iloc[0]
print(f"\nProbability mass on feasible plans (no repair needed): {feasible_mass:.1%}")
print(f"Best plan after validation/repair: {best.repaired_bits}  value = {best.repaired_value:.0f}"
      f"  (exact optimum = {optimal_value:.0f}, gap = {optimal_value - best.repaired_value:.0f})")

### Step 6: report - the shift plan

In [ ]:
plan_bits = best.repaired_bits
selected = [i for i, b in enumerate(plan_bits) if b == "1"]

print("=== Recommended maintenance plan for the next shift ===")
print(jobs.loc[selected, ["job", "value", "zone", "technician"]].to_string(index=False))
print(f"\nTotal value: {values[selected].sum():.0f} / exact optimum {optimal_value:.0f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ["tab:green" if i in selected else "lightgray" for i in G]
nx.draw_networkx(G, pos, ax=axes[0], node_color=colors, node_size=650,
                 labels={i: f"J{i}" for i in G}, font_size=9)
axes[0].set_title("Selected jobs (green) in the conflict graph")
axes[0].axis("off")

top = cand.sort_values("probs", ascending=False).head(10)
axes[1].bar(range(len(top)), top["probs"],
            color=["tab:green" if f else "tab:red" for f in top["feasible"]],
            tick_label=top["bits"])
axes[1].tick_params(axis="x", rotation=60)
axes[1].set_ylabel("Sampling probability")
axes[1].set_title("Most frequent samples (green = feasible, red = conflicts)")
fig.tight_layout()
plt.show()

## 3.4 Discussion: from the mini-pipeline to real projects

* **Penalty tuning.** If $P$ is too small, infeasible bitstrings can have lower cost than feasible ones; if it is too
  large, the diagonal (the actual objective) is drowned out and the hardware has a harder time.
* **Size.** 8 variables are easy for a laptop. The interest of neutral-atom hardware lies in larger,
  sparse, geometry-friendly instances; the number of atoms and the achievable layouts are limited by the device.
* **Hybrid workflow.** Quantum sampling proposes candidates; classical code validates, repairs and polishes them.
  The pipeline above is exactly that pattern.
* **Backends.** Switching from the local emulator to a remote emulator or a QPU only changes the backend object
  (and needs a Pasqal Cloud account); the modelling code stays the same.

**Where to go next**

* QoolQit: [documentation](https://docs.pasqal.com/qoolqit/) and [repository](https://github.com/pasqal-io/qoolqit)
* QUBO Solver: [documentation](https://pasqal-io.github.io/qubo-solver/latest/) and
  [repository](https://github.com/pasqal-io/qubo-solver) (embedding, drive shaping, backends, transforms)
* Pasqal Community Portal: <https://community.pasqal.com/>

**Challenges**

1. Add a third constraint type (e.g. "jobs 3 and 7 must not both be done because they share a spare part") and re-run.
2. Add a *budget* by penalising jobs' total duration with a quadratic term - it will add positive *and* dense
   off-diagonal terms; how does the embedding cope?
3. Swap for another embedding or change the drive-shaping method and compare the feasible probability mass.